# **ID Card OCR ICP Pipeline**


---



Multiple pipeline configurations, where each pipeline configuration provides some result to compare each configurations for ID card OCR

**Pipeline Stages**: image enhancement -> channel selection -> OCR



---


CS 8001 End to End Machine Learning

Ashley O'Gorman and Alexander Hardaway

# Setup
---

In [ ]:
import cv2
import numpy as np
import pytesseract
import easyocr
from PIL import Image
from datasets import load_dataset
import time
import pandas as pd
import re
import datetime
from collections import Counter
import random
import os
import json
import unicodedata
import glob
import io
import urllib.request
import torch
from minio import Minio
from concurrent.futures import ThreadPoolExecutor

In [ ]:
print("--- Loading dataset 'lansinuote/ocr_id_card' ---")
try:
    ds = load_dataset("lansinuote/ocr_id_card", split='train')
    print("Dataset loaded successfully.")
    print(ds)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

# Step 1: Image Enhancement Step


---


Contract: ***enhance_fn(image: np.ndarray) -> np.ndarray:***

The first step of the pipeline will be to use the cv2 library from python to enchance the base image if need be. The documentation to the library can be found here: https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html.

In [ ]:
# Baseline: no enhancement applied
def enhance_none(image: np.ndarray) -> np.ndarray:
    return image

Contrast

In [ ]:
# Ashley
def enhance_contrast_increase_25(image: np.ndarray) -> np.ndarray:
    contrast = 1.25
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_contrast_increase_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.5
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_contrast_increase_75(image: np.ndarray) -> np.ndarray:
    contrast = 1.75
    brightness = 0
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

Brightness

In [ ]:
# Ashley
def enhance_brightness_decrease_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = -50
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_brightness_increase_50(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = 50
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

In [ ]:
# Ashley
def enhance_brightness_increase_100(image: np.ndarray) -> np.ndarray:
    contrast = 1.0
    brightness = 100
    return cv2.convertScaleAbs(image, alpha=contrast, beta=brightness)

Laplacian

In [ ]:
# Ashley
def enhance_laplacian_simple(image: np.ndarray) -> np.ndarray:
    return cv2.Laplacian(image, -1)

In [ ]:
# Ashley
def enhance_laplacian_precise(image: np.ndarray) -> np.ndarray:
    laplacian_64f = cv2.Laplacian(image, cv2.CV_64F)
    laplacian_uint8 = cv2.convertScaleAbs(laplacian_64f)
    return laplacian_uint8

Saturation

In [ ]:
# Alex
def enhance_saturation_increase_50(image: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    s = cv2.convertScaleAbs(s, alpha=1.5, beta=0)
    return cv2.cvtColor(cv2.merge((h, s, v)), cv2.COLOR_HSV2RGB)

Sharpen

In [ ]:
# Alex
def enhance_sharpen(image: np.ndarray) -> np.ndarray:
    blurred = cv2.GaussianBlur(image, (0, 0), 3)
    return cv2.addWeighted(image, 1.5, blurred, -0.5, 0)

Deskew

In [ ]:
# Alex
def deskew_rotation(image: np.ndarray):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    card = cv2.inRange(hsv, (0, 0, 150), (180, 60, 255))
    contours, _ = cv2.findContours(card, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    _, _, angle = cv2.minAreaRect(max(contours, key=cv2.contourArea))
    if angle > 45:
        angle -= 90
    height, width = image.shape[:2]
    return cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)


def enhance_deskew(image: np.ndarray) -> np.ndarray:
    rotation = deskew_rotation(image)
    if rotation is None:
        return image
    height, width = image.shape[:2]
    return cv2.warpAffine(image, rotation, (width, height), borderMode=cv2.BORDER_REPLICATE)


def unrotate_boxes(result: list[dict], rotation: np.ndarray) -> list[dict]:
    inverse = cv2.invertAffineTransform(rotation)
    restored = []
    for r in result:
        x1, y1, x2, y2 = r["box"]
        corners = np.array([[x1, y1, 1], [x2, y1, 1], [x2, y2, 1], [x1, y2, 1]], dtype=float) @ inverse.T
        box = [corners[:, 0].min(), corners[:, 1].min(), corners[:, 0].max(), corners[:, 1].max()]
        restored.append({**r, "box": [round(float(v), 1) for v in box]})
    return restored

All enhancement options:

In [ ]:
enhancement_options = {
    "none": enhance_none,
    
    "+.25 contrast": enhance_contrast_increase_25,
    "+.50 contrast": enhance_contrast_increase_50,
    "+.75 contrast": enhance_contrast_increase_75,
    
    "-50 brightness": enhance_brightness_decrease_50,
    "+50 brightness": enhance_brightness_increase_50,
    "+100 brightness": enhance_brightness_increase_100,
    
    "simple laplacian": enhance_laplacian_simple,
    "precise laplacian": enhance_laplacian_precise,
    
    "+50% saturation": enhance_saturation_increase_50,
    "sharpen": enhance_sharpen,
    "deskew": enhance_deskew,
}

# Step 2: Channel Selection Step
---
Contract: ***channel_fn(image: np.ndarray) -> np.ndarray:***

The next step is choose to extract a specific color channel from the image array. One color or a combination of colors or the original image may be chosen. As before the cv2 library is used.

In [ ]:
# Baseline: no channel applied
def channel_none(image: np.ndarray) -> np.ndarray:
    return image

Single Colors

In [ ]:
# Ashley
def channel_blue(image: np.ndarray) -> np.ndarray:
    r, g, b = cv2.split(image)
    return b

In [ ]:
# Ashley
def channel_green(image: np.ndarray) -> np.ndarray:
    r, g, b = cv2.split(image)
    return g

In [ ]:
# Ashley
def channel_red(image: np.ndarray) -> np.ndarray:
    r, g, b = cv2.split(image)
    return r

Multicolors

In [ ]:
# Alex
def channel_red_green(image: np.ndarray) -> np.ndarray:
    return cv2.addWeighted(image[:, :, 0], 0.5, image[:, :, 1], 0.5, 0)

In [ ]:
# Alex
def channel_green_blue(image: np.ndarray) -> np.ndarray:
    return cv2.addWeighted(image[:, :, 1], 0.5, image[:, :, 2], 0.5, 0)

In [ ]:
# Alex
def channel_red_blue(image: np.ndarray) -> np.ndarray:
    return cv2.addWeighted(image[:, :, 0], 0.5, image[:, :, 2], 0.5, 0)

Grayscale

In [ ]:
# Alex
def channel_grayscale(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

All channel options

In [ ]:
channel_options = {
    "none": channel_none,

    "blue": channel_blue,
    "green": channel_green,
    "red": channel_red,
    
    "red-green": channel_red_green,
    "green-blue": channel_green_blue,
    "red-blue": channel_red_blue,

    "grayscale": channel_grayscale,
}

# Step 3: OCR Step

---
Contract: ***ocr_fn(image: np.ndarray) -> list[dict]:***

The final step is to actually perform actual ocr. For this step, we pass the processed numpy array of the input image to the ocr models. This uses the following OCR models:

EasyOcr

Tesseract

In [ ]:
# Ashley
def ocr_easyocr(image: np.ndarray) -> list[dict]:
    results = []
    for bbox, text, conf in easy_ocr_reader.readtext(image):
        xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
        results.append({"box": [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))],
                        "word": text, 
                        "conf": round(float(conf), 2)
                       })
    return results

In [ ]:
# Alex
TESSERACT_LANG = "chi_sim"
TESSERACT_PSM = 11

_CJK_SPACE = re.compile(r"(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])")

def ocr_tesseract(image: np.ndarray, psm: int = TESSERACT_PSM, tessdata_dir: str = None) -> list[dict]:
    data = pytesseract.image_to_data(
        image,
        lang=TESSERACT_LANG,
        config=f"--psm {psm}" + (f" --tessdata-dir {tessdata_dir}" if tessdata_dir else ""),
        output_type=pytesseract.Output.DICT,
    )
    lines = {}
    for i, word in enumerate(data["text"]):
        if not word.strip():
            continue
        key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
        x1, y1 = data["left"][i], data["top"][i]
        x2, y2 = x1 + data["width"][i], y1 + data["height"][i]
        if key not in lines:
            lines[key] = {"words": [], "box": [x1, y1, x2, y2], "confs": []}
        line = lines[key]
        line["words"].append(word)
        line["box"] = [
            min(line["box"][0], x1),
            min(line["box"][1], y1),
            max(line["box"][2], x2),
            max(line["box"][3], y2),
        ]
        line["confs"].append(float(data["conf"][i]))

    results = []
    for line in lines.values():
        results.append({
            "box": line["box"],
            "word": _CJK_SPACE.sub("", " ".join(line["words"])),
            "conf": round(sum(line["confs"]) / len(line["confs"]) / 100, 2),
        })
    return results

All OCR options:

In [ ]:
ocr_options = {
    "tesseract": ocr_tesseract,
    "easyocr": ocr_easyocr,
}

# Assertion


---
Check the accuracy of the pipeline.


Milestone 1 assertions and field check, kept to report the Milestone 1 accuracy definition next to the Milestone 2 one.

In [ ]:
# Alex
FIELD_NAMES = ["name", "sex", "ethnicity", "year", "month", "day", "address", "id_number"]

def ocr_lines(result) -> list[str]:
    lines = []
    for r in result:
        if isinstance(r, dict):
            lines.append(r["word"])
        elif isinstance(r, str):
            lines.append(r)
        else:
            lines.append(r[1])
    return lines


def find_id_number(lines: list[str]):
    for line in lines:
        match = re.search(r"(?<!\d)\d{17}[\dXx](?!\d)", "".join(line.split()))
        if match:
            return match.group()
    return None


def find_date(lines: list[str]):
    numbers = [int(n) for line in lines if find_id_number([line]) is None for n in re.findall(r"\d+", line)]
    for i in range(len(numbers) - 2):
        if 1900 <= numbers[i] <= 2030:
            try:
                return datetime.date(numbers[i], numbers[i + 1], numbers[i + 2])
            except (ValueError, OverflowError):
                return None
    return None


def run_assertions(result) -> dict:
    lines = ocr_lines(result)
    text = "".join("".join(lines).split())
    id_number = find_id_number(lines)
    date = find_date(lines)
    return {
        "date_valid": date is not None,
        "id_format": id_number is not None,
        "id_matches_date": id_number is not None and date is not None and id_number[6:14] == date.strftime("%Y%m%d"),
        "sex_valid": ("男" in text) != ("女" in text),
    }


def check_fields(result, target) -> dict:
    lines = ocr_lines(result)
    text = "".join("".join(lines).split())
    lines_without_id = [line for line in lines if find_id_number([line]) is None]
    numbers = Counter(int(n) for line in lines_without_id for n in re.findall(r"\d+", line))
    fields = {}
    for t in target:
        name = FIELD_NAMES[t["cls"]]
        value = "".join(t["word"].split())
        if name in ("year", "month", "day"):
            fields[name] = numbers[int(value)] > 0
            if fields[name]:
                numbers[int(value)] -= 1
        else:
            fields[name] = value in text
    return fields

# Evaluation harness


---
Slices, metric, and assertions as defined in the Pipeline Evaluation Guide (ID card OCR section).

In [ ]:
# Alex
DATASET_NAME = "lansinuote/ocr_id_card"
DATASET_REVISION = "6a7fd6ba549e99b9a3517fc9a56c4e3377e43e05"

order = np.random.default_rng(8001).permutation(len(ds))
dev_ids = [int(i) for i in order[:500]]
test_ids = [int(i) for i in order[500:2500]]
load_pool_ids = [int(i) for i in order[2500:]]

os.makedirs("results/splits", exist_ok=True)
with open("results/splits/splits.json", "w") as f:
    json.dump({
        "dataset": DATASET_NAME,
        "revision": DATASET_REVISION,
        "generator": "numpy.random.default_rng(8001).permutation(20000)",
        "dev": dev_ids,
        "test": test_ids,
        "load_pool": load_pool_ids,
    }, f)

all_labels = ds.select_columns(["ocr"])["ocr"]
print(f"dev {len(dev_ids)}, test {len(test_ids)}, load pool {len(load_pool_ids)}")

In [ ]:
# Alex
FIELD_CLASSES = ["name", "gender", "ethnicity", "year", "month", "day", "address", "id_number"]


def normalize(text: str) -> str:
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", text)).replace("x", "X")


def box_overlap(a, b) -> float:
    width = min(a[2], b[2]) - max(a[0], b[0])
    height = min(a[3], b[3]) - max(a[1], b[1])
    if width <= 0 or height <= 0:
        return 0.0
    smaller = min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
    return width * height / smaller if smaller > 0 else 0.0


def reading_order(detections: list[dict]) -> list[dict]:
    lines = []
    for r in sorted(detections, key=lambda r: (r["box"][1] + r["box"][3]) / 2):
        center = (r["box"][1] + r["box"][3]) / 2
        if lines and abs(center - lines[-1][0]) <= (r["box"][3] - r["box"][1]) / 2:
            lines[-1][1].append(r)
        else:
            lines.append((center, [r]))
    return [r for _, line in lines for r in sorted(line, key=lambda r: r["box"][0])]


def assign_fields(result: list[dict], target: list[dict]) -> dict:
    assigned = {t["cls"]: [] for t in target}
    for r in result:
        best_overlap, best_cls = max((box_overlap(r["box"], t["box"]), t["cls"]) for t in target)
        if best_overlap >= 0.5:
            assigned[best_cls].append(r)
    return {FIELD_CLASSES[cls]: normalize("".join(r["word"] for r in reading_order(detections))) for cls, detections in assigned.items()}


def edit_distance(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def score_fields(fields: dict, target: list[dict]) -> dict:
    scores = {}
    for t in target:
        name = FIELD_CLASSES[t["cls"]]
        truth = normalize(t["word"])
        if not truth:
            continue
        scores[f"{name}_correct"] = fields[name] == truth
        scores[f"{name}_cer"] = edit_distance(fields[name], truth) / len(truth)
    return scores

In [ ]:
# Alex
assertion_names = ["A1_id_format", "A2_id_matches_date", "A3_valid_date", "A4_valid_gender"]


def required_assertions(fields: dict) -> dict:
    id_number = fields.get("id_number", "")
    year, month, day = fields.get("year", ""), fields.get("month", ""), fields.get("day", "")
    id_format = re.fullmatch(r"\d{17}[\dX]", id_number) is not None
    date_digits = all(re.fullmatch(r"\d+", v) for v in (year, month, day))
    try:
        valid_date = date_digits and datetime.date(1900, 1, 1) <= datetime.date(int(year), int(month), int(day)) <= datetime.date(2030, 12, 31)
    except (ValueError, OverflowError):
        valid_date = False
    return {
        "A1_id_format": id_format,
        "A2_id_matches_date": id_format and date_digits and id_number[6:14] == year + month.zfill(2) + day.zfill(2),
        "A3_valid_date": valid_date,
        "A4_valid_gender": fields.get("gender", "") in ("男", "女"),
    }

Required assertions checked on the ground-truth labels of all 20,000 cards.

In [ ]:
# Alex
truth_rows = []
for card_id, labels in enumerate(all_labels):
    truth_fields = {FIELD_CLASSES[t["cls"]]: normalize(t["word"]) for t in labels}
    truth_rows.append({"card": card_id, **truth_fields, **required_assertions(truth_fields)})
ground_truth = pd.DataFrame(truth_rows)

ground_truth_table = pd.DataFrame({
    "cards passing": ground_truth[assertion_names].sum(),
    "cards failing": (~ground_truth[assertion_names]).sum(),
    "pass rate (%)": (100 * ground_truth[assertion_names].mean()).round(2),
})
os.makedirs("results", exist_ok=True)
ground_truth_table.to_csv("results/ground_truth_assertions.csv")
invalid_dates = ground_truth.loc[~ground_truth["A3_valid_date"], ["card", "year", "month", "day", "id_number"]]
invalid_dates.to_csv("results/ground_truth_invalid_dates.csv", index=False)
print(f"Cards whose labels fail A3: {len(invalid_dates)}")
ground_truth_table

# Pipeline


---
Method to run each pipeline.


In [ ]:
# Ashley
def run_pipeline(image, enhance_fn, channel_fn, ocr_fn):
    start = time.perf_counter()
    x = enhance_fn(image)
    x = channel_fn(x)
    img_text = ocr_fn(x)
    latency = time.perf_counter() - start
    return img_text, latency

# Run configurations


---
Each configuration is run on a list of cards. Raw OCR output is cached in results/ocr_cache so scoring can be rerun without rerunning OCR.

In [ ]:
# This can take a while, only needed once
easy_ocr_reader = easyocr.Reader(['ch_sim'], gpu=False) #KVM@TACC is cpu only

In [ ]:
# Alex
os.environ["OMP_THREAD_LIMIT"] = "1"
TESSERACT_WORKERS = max(1, (os.cpu_count() or 2) - 2)
EASYOCR_WORKERS = max(1, (os.cpu_count() or 2) // 2)
if EASYOCR_WORKERS > 1:
    torch.set_num_threads(2)
CHUNK_SIZE = 64


def ocr_tesseract_default(image: np.ndarray) -> list[dict]:
    return ocr_tesseract(image, psm=3)


eval_ocr_options = {**ocr_options, "tesseract (default)": ocr_tesseract_default}
BASELINE = ("none", "none", "tesseract (default)")


def run_card(image, e_name, c_name, o_name):
    result, latency = run_pipeline(image, enhancement_options[e_name], channel_options[c_name], eval_ocr_options[o_name])
    if e_name == "deskew":
        rotation = deskew_rotation(image)
        if rotation is not None:
            result = unrotate_boxes(result, rotation)
    return result, latency


def run_configs(configs, card_ids, slice_name) -> list[dict]:
    cache_path = f"results/ocr_cache/{slice_name}.jsonl"
    os.makedirs("results/ocr_cache", exist_ok=True)
    done = {}
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                done[(row["enhancement"], row["channel"], row["ocr"], row["card"])] = row
    for n, (e_name, c_name, o_name) in enumerate(configs, 1):
        todo = [i for i in card_ids if (e_name, c_name, o_name, i) not in done]
        print(f"[{n}/{len(configs)}] {e_name} + {c_name} + {o_name}: {len(todo)} of {len(card_ids)} cards to run")
        workers = TESSERACT_WORKERS if o_name.startswith("tesseract") else EASYOCR_WORKERS
        for start in range(0, len(todo), CHUNK_SIZE):
            chunk = todo[start:start + CHUNK_SIZE]
            images = [np.array(ds[i]["image"]) for i in chunk]
            with ThreadPoolExecutor(workers) as pool:
                outputs = list(pool.map(lambda image: run_card(image, e_name, c_name, o_name), images))
            with open(cache_path, "a", encoding="utf-8") as f:
                for card_id, (result, latency) in zip(chunk, outputs):
                    row = {"enhancement": e_name, "channel": c_name, "ocr": o_name, "card": card_id, "latency": latency, "result": result}
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    done[(e_name, c_name, o_name, card_id)] = row
    return [done[(e, c, o, i)] for e, c, o in configs for i in card_ids]

In [ ]:
# Alex
def score_runs(runs: list[dict]) -> pd.DataFrame:
    rows = []
    for run in runs:
        target = all_labels[run["card"]]
        fields = assign_fields(run["result"], target)
        scores = score_fields(fields, target)
        assertions = required_assertions(fields)
        correct = [v for k, v in scores.items() if k.endswith("_correct")]
        rows.append({
            "enhancement": run["enhancement"],
            "channel": run["channel"],
            "ocr": run["ocr"],
            "card": run["card"],
            "fields_correct": sum(correct),
            "fields_scored": len(correct),
            "card_exact": all(correct),
            "mean_cer": np.mean([v for k, v in scores.items() if k.endswith("_cer")]),
            "all_assertions": all(assertions.values()),
            "m1_assertions_passed": sum(run_assertions(run["result"]).values()),
            "m1_fields_correct": sum(check_fields(run["result"], target).values()),
            **scores,
            **assertions,
        })
    return pd.DataFrame(rows)


def summarize(card_scores: pd.DataFrame) -> pd.DataFrame:
    grouped = card_scores.groupby(["enhancement", "channel", "ocr"], sort=False)
    table = pd.DataFrame({
        "cards": grouped.size(),
        "field exact match (%)": 100 * grouped["fields_correct"].sum() / grouped["fields_scored"].sum(),
        "card exact match (%)": 100 * grouped["card_exact"].mean(),
        "mean CER": grouped["mean_cer"].mean(),
        "assertion pass rate (%)": 100 * grouped["all_assertions"].mean(),
        **{f"{a} (%)": 100 * grouped[a].mean() for a in assertion_names},
        "M1 accuracy (%)": 100 * grouped["m1_assertions_passed"].sum() / (4 * grouped.size()),
        "M1 fields correct (%)": 100 * grouped["m1_fields_correct"].sum() / (8 * grouped.size()),
    }).round(2).reset_index()
    return table.sort_values("field exact match (%)", ascending=False, ignore_index=True)


def per_field_table(card_scores: pd.DataFrame) -> pd.DataFrame:
    grouped = card_scores.groupby(["enhancement", "channel", "ocr"], sort=False)
    table = pd.DataFrame({f"{name} exact (%)": 100 * grouped[f"{name}_correct"].mean() for name in FIELD_CLASSES})
    for name in FIELD_CLASSES:
        table[f"{name} CER"] = grouped[f"{name}_cer"].mean()
    return table.round(2).reset_index()

# Development slice: screen all Milestone 1 configurations


---
All 192 Milestone 1 configurations plus the baseline run on all 500 cards of the development slice. The configurations carried forward to the test slice are chosen here: the best CARRY_TOP per OCR engine, the no-preprocessing configuration of each engine, and CARRY_SPREAD more per engine spread evenly down the ranking so the test slice also covers weak configurations.

In [ ]:
# Alex
CARRY_TOP = 3
CARRY_SPREAD = 3

m1_configs = [(e, c, o) for e in enhancement_options for c in channel_options for o in ocr_options]
total_configs = len(m1_configs)
print(f"Number of distinct pipeline configurations: {len(enhancement_options)} enhancement x {len(channel_options)} channel x {len(ocr_options)} OCR = {total_configs}")

dev_runs = run_configs(m1_configs + [BASELINE], dev_ids, "dev")
dev_scores = score_runs(dev_runs)
dev_table = summarize(dev_scores)
dev_table.to_csv("results/dev_table.csv", index=False)

carried = [BASELINE]
for o_name in ocr_options:
    ranked = [tuple(r) for r in dev_table.loc[dev_table["ocr"] == o_name, ["enhancement", "channel", "ocr"]].values]
    spread = np.linspace(CARRY_TOP, len(ranked) - 1, CARRY_SPREAD + 1).round().astype(int)[1:]
    carried += ranked[:CARRY_TOP] + [("none", "none", o_name)] + [ranked[i] for i in spread]
carried = list(dict.fromkeys(carried))
pd.set_option("display.max_rows", None)
display(dev_table)
pd.DataFrame(carried, columns=["enhancement", "channel", "ocr"])

# Test slice results


---
Carried-forward configurations on all 2,000 test cards. Only these numbers are reported.

In [ ]:
# Alex
test_runs = run_configs(carried, test_ids, "test")
test_scores = score_runs(test_runs)
test_table = summarize(test_scores)
test_table.insert(3, "baseline", [tuple(r) == BASELINE for r in test_table[["enhancement", "channel", "ocr"]].values])
test_table.to_csv("results/test_table.csv", index=False)
test_scores.to_csv("results/test_card_scores.csv", index=False)
per_field_table(test_scores).to_csv("results/test_per_field.csv", index=False)
test_table

In [ ]:
# Alex
per_field_table(test_scores)

Assertion pass rate compared with offline accuracy on the test slice.

In [ ]:
# Alex
accuracy_col, assertion_col = "field exact match (%)", "assertion pass rate (%)"
ranking = test_table[["enhancement", "channel", "ocr", accuracy_col, assertion_col]].copy()
ranking["rank by accuracy"] = ranking[accuracy_col].rank(ascending=False, method="min").astype(int)
ranking["rank by assertions"] = ranking[assertion_col].rank(ascending=False, method="min").astype(int)
ranking.to_csv("results/test_assertion_vs_accuracy.csv", index=False)

agreement = pd.DataFrame({
    "configurations": [len(ranking)],
    "pearson": [ranking[assertion_col].corr(ranking[accuracy_col])],
    "spearman": [ranking[assertion_col].corr(ranking[accuracy_col], method="spearman")],
}).round(3)

card_agreement = pd.crosstab(
    test_scores["all_assertions"].map({True: "passes all assertions", False: "fails an assertion"}),
    test_scores["card_exact"].map({True: "all 8 fields correct", False: "some field wrong"}),
)
card_agreement.to_csv("results/test_card_agreement.csv")
display(agreement)
display(card_agreement)
ranking

Stage contribution: OCR accuracy on the test slice with and without the preprocessing chosen on the development slice, using the same OCR variant.

In [ ]:
# Alex
contribution = []
for o_name in ocr_options:
    best = dev_table[dev_table["ocr"] == o_name].iloc[0]
    with_pre = test_table[(test_table["enhancement"] == best["enhancement"]) & (test_table["channel"] == best["channel"]) & (test_table["ocr"] == o_name)].iloc[0]
    without = test_table[(test_table["enhancement"] == "none") & (test_table["channel"] == "none") & (test_table["ocr"] == o_name)].iloc[0]
    contribution.append({
        "ocr": o_name,
        "preprocessing": f"{best['enhancement']} + {best['channel']}",
        "field exact match with (%)": with_pre[accuracy_col],
        "field exact match without (%)": without[accuracy_col],
        "difference (points)": round(with_pre[accuracy_col] - without[accuracy_col], 2),
    })
contribution = pd.DataFrame(contribution)
contribution.to_csv("results/test_stage_contribution.csv", index=False)
contribution

# Model variants (T2)


---
Variants of the OCR stage, the only scalable stage. Each variant is run on the test slice with the preprocessing chosen for its engine on the development slice, then stored in the MinIO model registry with its metadata.

In [ ]:
# Alex
MODEL_DIR = "models"
TESSDATA_SOURCES = {
    "fast": "https://github.com/tesseract-ocr/tessdata_fast/raw/main/chi_sim.traineddata",
    "best": "https://github.com/tesseract-ocr/tessdata_best/raw/main/chi_sim.traineddata",
}
tessdata_dirs = {}
for kind, url in TESSDATA_SOURCES.items():
    folder = os.path.join(MODEL_DIR, f"tessdata_{kind}")
    path = os.path.join(folder, "chi_sim.traineddata")
    if not os.path.exists(path):
        os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    tessdata_dirs[kind] = folder
system_tessdata = os.environ.get("TESSDATA_PREFIX") or next(iter(glob.glob("/usr/share/tesseract-ocr/*/tessdata")), "")
tessdata_dirs["system"] = system_tessdata
{kind: os.path.getsize(os.path.join(folder, "chi_sim.traineddata")) for kind, folder in tessdata_dirs.items()}

In [ ]:
# Alex
easy_ocr_reader_fp32 = easyocr.Reader(["ch_sim"], gpu=False, quantize=False)


def easyocr_with(reader, **options):
    def ocr(image: np.ndarray) -> list[dict]:
        results = []
        for bbox, text, conf in reader.readtext(image, **options):
            xs = [p[0] for p in bbox]; ys = [p[1] for p in bbox]
            results.append({"box": [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))], "word": text, "conf": round(float(conf), 2)})
        return results
    return ocr


def tesseract_with(kind: str, psm: int = TESSERACT_PSM):
    def ocr(image: np.ndarray) -> list[dict]:
        return ocr_tesseract(image, psm=psm, tessdata_dir=tessdata_dirs[kind])
    return ocr


def resized(ocr_fn, scale: float):
    def ocr(image: np.ndarray) -> list[dict]:
        scaled = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        return [{**r, "box": [round(v / scale, 1) for v in r["box"]]} for r in ocr_fn(scaled)]
    return ocr


def split_runs(ocr_fn):
    def ocr(image: np.ndarray) -> list[dict]:
        pieces = []
        for r in ocr_fn(image):
            tokens = re.findall(r"\d[\dXx]*|\S", r["word"])
            if not tokens:
                continue
            widths = [sum(0.5 if unicodedata.east_asian_width(ch) in ("Na", "H") else 1.0 for ch in t) for t in tokens]
            x1, y1, x2, y2 = r["box"]
            step = (x2 - x1) / sum(widths)
            for token, width in zip(tokens, widths):
                pieces.append({"box": [round(x1, 1), y1, round(x1 + width * step, 1), y2], "word": token, "conf": r["conf"]})
                x1 += width * step
        return pieces
    return ocr


tessdata_file = {kind: os.path.join(folder, "chi_sim.traineddata") for kind, folder in tessdata_dirs.items()}
easyocr_files = [os.path.expanduser(f"~/.EasyOCR/model/{f}") for f in ("craft_mlt_25k.pth", "zh_sim_g2.pth")]
ocr_variants = {
    "tesseract": {"fn": ocr_tesseract, "engine": "tesseract", "files": [tessdata_file["system"]],
                  "source": "chi_sim.traineddata installed with Tesseract", "config": {"psm": TESSERACT_PSM, "tessdata": "system"}},
    "tesseract-fast": {"fn": tesseract_with("fast"), "engine": "tesseract", "files": [tessdata_file["fast"]],
                       "source": TESSDATA_SOURCES["fast"], "config": {"psm": TESSERACT_PSM, "tessdata": "fast"}},
    "tesseract-best": {"fn": tesseract_with("best"), "engine": "tesseract", "files": [tessdata_file["best"]],
                       "source": TESSDATA_SOURCES["best"], "config": {"psm": TESSERACT_PSM, "tessdata": "best"}},
    "tesseract-2x": {"fn": resized(ocr_tesseract, 2.0), "engine": "tesseract", "files": [tessdata_file["system"]],
                     "source": "chi_sim.traineddata installed with Tesseract", "config": {"psm": TESSERACT_PSM, "tessdata": "system", "input_scale": 2.0}},
    "tesseract-runs": {"fn": split_runs(ocr_tesseract), "engine": "tesseract", "files": [tessdata_file["system"]],
                       "source": "chi_sim.traineddata installed with Tesseract", "config": {"psm": TESSERACT_PSM, "tessdata": "system", "split": "digit runs and single characters"}},
    "easyocr": {"fn": ocr_easyocr, "engine": "easyocr", "files": easyocr_files,
                "source": "EasyOCR 1.6.2 ch_sim, dynamic int8 quantization at load (EasyOCR CPU default)", "config": {"quantize": True}},
    "easyocr-fp32": {"fn": easyocr_with(easy_ocr_reader_fp32), "engine": "easyocr", "files": easyocr_files,
                     "source": "EasyOCR 1.6.2 ch_sim, full precision", "config": {"quantize": False}},
    "easyocr-runs": {"fn": split_runs(easyocr_with(easy_ocr_reader, width_ths=0.0)), "engine": "easyocr", "files": easyocr_files,
                     "source": "EasyOCR 1.6.2 ch_sim, dynamic int8 quantization at load (EasyOCR CPU default)", "config": {"quantize": True, "width_ths": 0.0, "split": "digit runs and single characters"}},
}
eval_ocr_options.update({name: v["fn"] for name, v in ocr_variants.items()})

In [ ]:
# Alex
best_preprocessing = {o_name: tuple(dev_table[dev_table["ocr"] == o_name].iloc[0][["enhancement", "channel"]]) for o_name in ocr_options}
variant_configs = [(*best_preprocessing[v["engine"]], name) for name, v in ocr_variants.items()]

variant_runs = run_configs(variant_configs, test_ids, "test")
variant_scores = score_runs(variant_runs)
variant_table = summarize(variant_scores)
variant_table.to_csv("results/test_variant_table.csv", index=False)
variant_table

In [ ]:
# Alex
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin")
REGISTRY_BUCKET = "icp-ocr-variants"
VARIANT_VERSION = "1"

registry_rows = []
for name, v in ocr_variants.items():
    row = variant_table[variant_table["ocr"] == name].iloc[0]
    registry_rows.append({
        "name": name,
        "version": VARIANT_VERSION,
        "engine": v["engine"],
        "source": v["source"],
        "config": json.dumps(v["config"]),
        "files": json.dumps([os.path.basename(p) for p in v["files"]]),
        "size_on_disk_bytes": sum(os.path.getsize(p) for p in v["files"]),
        "preprocessing": f"{row['enhancement']} + {row['channel']}",
        "test field exact match (%)": row["field exact match (%)"],
        "test assertion pass rate (%)": row["assertion pass rate (%)"],
    })
registry = pd.DataFrame(registry_rows)
registry.to_csv("results/variant_registry.csv", index=False)

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(REGISTRY_BUCKET):
    client.make_bucket(REGISTRY_BUCKET)
for record, (name, v) in zip(registry_rows, ocr_variants.items()):
    prefix = f"{name}/{VARIANT_VERSION}"
    for path in v["files"]:
        client.fput_object(REGISTRY_BUCKET, f"{prefix}/{os.path.basename(path)}", path)
    metadata = json.dumps({**record, "config": v["config"]}, ensure_ascii=False).encode("utf-8")
    client.put_object(REGISTRY_BUCKET, f"{prefix}/metadata.json", io.BytesIO(metadata), len(metadata), content_type="application/json")
print(sorted(o.object_name for o in client.list_objects(REGISTRY_BUCKET, recursive=True)))
registry